In [1]:
"""
Ce script construit un modèle pour classifier automatiquement les commentaires des pharmaciens
en 11 classes principales de problèmes médicamenteux.
Préparé par Anissa OUAREM
"""

# Étape 1 : Importer les bibliothèques nécessaires
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# --- Début de la configuration ---

# Étape 2 : Charger les données depuis le fichier CSV
# Assurez-vous d'avoir téléversé le fichier "data_defi3.csv" dans votre environnement Google Colab
file_path = 'data_defi3.csv'
# On lit le fichier en spécifiant le séparateur ";"
try:
    data = pd.read_csv(file_path, sep=';')
except FileNotFoundError:
    print(f"Erreur : Le fichier '{file_path}' n'a pas été trouvé. Veuillez vous assurer qu'il est correctement chargé.")
    exit()

# Étape 3 : Préparer les données
# Nettoyage de base : supprimer les lignes sans commentaire ou sans classe
data.dropna(subset=['Avis.Pharmaceutique', 'PLT'], inplace=True)

# Extraire la classe principale (la partie entière du nombre)
# Exemple : '6.3' -> 6, '4.1' -> 4
# D'abord, on remplace la virgule (,) par un point (.) pour unifier le format
# Ensuite, on convertit en numérique, en ignorant les erreurs de conversion
# Finalement, on convertit en entier
data['main_class'] = pd.to_numeric(data['PLT'].astype(str).str.replace(',', '.'), errors='coerce')
data.dropna(subset=['main_class'], inplace=True) # Supprimer les lignes qui n'ont pas pu être converties
data['main_class'] = data['main_class'].astype(int)


# Définir les variables d'entrée (X) et la cible (y)
X = data['Avis.Pharmaceutique'] # Les commentaires
y = data['main_class']           # La classe principale


# --- Construction du modèle ---

# Étape 4 : Diviser les données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Étape 5 : Transformer le texte en vecteurs numériques (Vectorisation)
# On utilise à nouveau TfidfVectorizer
# Vous pouvez ajouter des stop words français ici, par exemple: stop_words='french'
vectorizer = TfidfVectorizer(max_features=5000, stop_words=None)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


# Étape 6 : Entraîner le modèle de classification
# On utilise LinearSVC, un modèle robuste et efficace pour la classification de texte
model = LinearSVC(random_state=42, C=0.5)
model.fit(X_train_vec, y_train)


# --- Évaluation du modèle ---

# Étape 7 : Prédiction et évaluation des performances
y_pred = model.predict(X_test_vec)

# Afficher le rapport de performance
print("--- Rapport de performance du modèle de classification multi-classe ---")
# On récupère la liste des classes uniques dans les données de test pour s'assurer que le rapport les affiche correctly
unique_labels = sorted(y_test.unique())
target_names = [f'Classe {label}' for label in unique_labels]
print(classification_report(y_test, y_pred, labels=unique_labels, target_names=target_names))


# --- Exemple d'utilisation ---
print("\n--- Test du modèle sur un nouveau commentaire ---")
new_comment = "posologie infraT veuillez réévaluer la posologie"
new_comment_vec = vectorizer.transform([new_comment])
prediction = model.predict(new_comment_vec)

print(f"Commentaire : '{new_comment}'")
print(f"Résultat : Classifié dans la classe principale n° {prediction[0]}")

--- Rapport de performance du modèle de classification multi-classe ---
              precision    recall  f1-score   support

    Classe 1       0.86      0.92      0.89      2253
    Classe 2       0.84      0.73      0.78       151
    Classe 3       0.45      0.28      0.35       194
    Classe 4       0.77      0.76      0.76       613
    Classe 5       0.74      0.61      0.67       166
    Classe 6       0.86      0.85      0.85       193
    Classe 7       0.00      0.00      0.00         2
    Classe 8       0.76      0.81      0.78       589
    Classe 9       1.00      0.75      0.86         4
   Classe 10       0.76      0.70      0.73       148
   Classe 11       0.64      0.57      0.61       316

    accuracy                           0.80      4629
   macro avg       0.70      0.64      0.66      4629
weighted avg       0.80      0.80      0.80      4629


--- Test du modèle sur un nouveau commentaire ---
Commentaire : 'posologie infraT veuillez réévaluer la posologie'

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
